<a href="https://colab.research.google.com/github/rajkumark30/Agentic_AI_Training/blob/main/Capstone_Project_Rajkumar_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Capstone Project - Document Processing Agent
User: Rajkumar

Requirements:
- OCR for scanned PDFs
- Classification + Extraction using LLM
- Structured JSON output
- Human-in-the-loop
- LangGraph workflow


In [1]:
!apt-get install -y tesseract-ocr poppler-utils
!pip install langchain==0.2.16 langgraph pypdf pytesseract pdf2image pillow openai


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.


In [2]:
import os
from openai import OpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph
from typing import TypedDict
from google.colab import files
from pdf2image import convert_from_path
import pytesseract
from pypdf import PdfReader


In [3]:
client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key='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'
)


In [4]:
uploaded = files.upload()


Saving bw_doc_1.pdf to bw_doc_1 (1).pdf
Saving bw_doc_2.pdf to bw_doc_2 (1).pdf
Saving bw_doc_3.pdf to bw_doc_3 (1).pdf
Saving bw_doc_4.pdf to bw_doc_4 (1).pdf
Saving bw_doc_5.pdf to bw_doc_5 (1).pdf
Saving LoA1.pdf to LoA1 (1).pdf
Saving LOA2.pdf to LOA2 (1).pdf
Saving LOA3.pdf to LOA3 (1).pdf
Saving LOA4.pdf to LOA4 (1).pdf
Saving LOA5.pdf to LOA5 (1).pdf
Saving LOA6.pdf to LOA6 (1).pdf
Saving LOA7.pdf to LOA7 (1).pdf
Saving LOA8.pdf to LOA8 (1).pdf
Saving LOA9.pdf to LOA9 (1).pdf
Saving notice_1.pdf to notice_1 (1).pdf
Saving notice_2.pdf to notice_2 (1).pdf
Saving notice_3.pdf to notice_3 (1).pdf
Saving notice_4.pdf to notice_4 (1).pdf
Saving notice_5.pdf to notice_5 (1).pdf


In [5]:
def extract_text(file_path):
    try:
        reader = PdfReader(file_path)
        text = ''
        for page in reader.pages:
            text += page.extract_text() or ''
        if len(text.strip()) > 50:
            return text
    except:
        pass
    images = convert_from_path(file_path, dpi=100, first_page=1, last_page=2)
    text = ''
    for img in images:
        text += pytesseract.image_to_string(img)
    return text

documents = []
for file in uploaded.keys():
    print(f'Processing: {file}')
    text = extract_text(file)
    documents.append({'file': file, 'text': text})

print(f'Processed {len(documents)} documents')


Processing: bw_doc_1 (1).pdf
Processing: bw_doc_2 (1).pdf
Processing: bw_doc_3 (1).pdf
Processing: bw_doc_4 (1).pdf
Processing: bw_doc_5 (1).pdf
Processing: LoA1 (1).pdf
Processing: LOA2 (1).pdf
Processing: LOA3 (1).pdf
Processing: LOA4 (1).pdf
Processing: LOA5 (1).pdf
Processing: LOA6 (1).pdf
Processing: LOA7 (1).pdf
Processing: LOA8 (1).pdf
Processing: LOA9 (1).pdf
Processing: notice_1 (1).pdf
Processing: notice_2 (1).pdf
Processing: notice_3 (1).pdf
Processing: notice_4 (1).pdf
Processing: notice_5 (1).pdf
Processed 19 documents


In [6]:
class State(TypedDict):
    file: str
    text: str
    result: str
    approved: bool


In [7]:
def process_doc(state):
    prompt = f"""
Return ONLY valid JSON:

{{
  "doc_type": "",
  "parties": [],
  "date": "",
  "summary": "",
  "confidence": 0
}}

Text:
{state['text'][:3000]}
"""

    response = client.chat.completions.create(
        model="openai/gpt-3.5-turbo",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    state["result"] = response.choices[0].message.content
    return state

In [8]:
def review(state):
    print(f"\nFile: {state['file']}")
    print(state['result'])
    decision = input('Approve? (yes/no): ')
    state['approved'] = decision.lower() == 'yes'
    return state


In [9]:
builder = StateGraph(State)
builder.add_node('process', process_doc)
builder.add_node('review', review)
builder.set_entry_point('process')
builder.add_edge('process', 'review')
graph = builder.compile()


In [10]:
results = []
for doc in documents:
    state = {'file': doc['file'], 'text': doc['text'], 'result': '', 'approved': False}
    results.append(graph.invoke(state))
results



File: bw_doc_1 (1).pdf
{
  "doc_type": "Notice",
  "parties": ["Abernathy & Rowe", "Client Affairs"],
  "date": "2022-01-15",
  "summary": "Notice Regarding Limited Power of Attorney Representation",
  "confidence": 0.8
}
Approve? (yes/no): yes

File: bw_doc_2 (1).pdf
{
  "doc_type": "",
  "parties": [],
  "date": "",
  "summary": "",
  "confidence": 0
}
Approve? (yes/no): no

File: bw_doc_3 (1).pdf
{
  "doc_type": "Request for Limited Document Production for Audit",
  "parties": [],
  "date": "",
  "summary": "Our office presents a corporate client engaged in a routine compliance audit that necessitates limited access to historical account correspondence and transaction logs within a defined timeframe.",
  "confidence": 0
}
Approve? (yes/no): no

File: bw_doc_4 (1).pdf
{
  "doc_type": "Guardianship Consultation Request",
  "parties": ["Grafton & Meyers, Family Law Division", "Path Howe, Family Row 'owconamns"],
  "date": "",
  "summary": "Request for guardianship consultation on beha

[{'file': 'bw_doc_1 (1).pdf',
  'text': "Abernathy & Rowe - Client Affairs ae nnoscas0\n\n105 Meeitan Aves, Sate 210\n\nRecords Coordinator\n\nNotice Regarding Limited Power of Atorney Representation\n\n \n\nWe represent an individual referred to inthis correspondence as the Principal We write\n‘advise thatthe Principal has exocuted a limited Power af Attornoy in Favor of our\n\nclient, who will act on tho Principal's behelf fora narrowly specif set of\n\n‘administrative and transactional matters concerning account administration and authorized\ntransfers. Tho purpose of this notco is informational: o notify your office of a valid\nInstrument granting authoriy to our client to view account stamens, submis\ndocumented instructions limited to aecount reconciliation, and to aso with your waft\n\non diserte procedural issues. This communication doesnot ator or expand any contract\nobligations you maintain withthe Prineipa, nor does it constitate an imevocable\n\nAireetive to change account